# Inference notebook: U-Net from scratch (first official submission)
**SurfaceGuard AI.** Loads the checkpoint trained in the Milestone 2 notebook (`m2-unet-scratch-v1`), predicts every test image, thresholds at the best validation threshold (0.8), RLE-encodes the masks and writes `submission.csv` in the exact row order of `sample_submission.csv`.

## 1. Setup and constants

In [1]:
import os, glob, random, time
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from PIL import Image

SEED = 42
def seed_everything(seed=SEED):
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
seed_everything()

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("PyTorch", torch.__version__, "| device:", DEVICE)

IMG_H, IMG_W = 128, 800
CANVAS_PIXELS = IMG_H * IMG_W
DEFECT_TYPES = ["defect_a", "defect_b", "defect_c", "defect_d"]

PyTorch 2.11.0+cu128 | device: cuda


In [2]:
# Values taken from the training run m2-unet-scratch-v1 (W&B run 34c5iose)
IMG_MEAN, IMG_STD = 0.3314, 0.1892      # training-set pixel statistics used during training
BEST_TAU = 0.8                          # best validation threshold (score 0.6028)
DEBUG = False

## 2. RLE encoder
1-based, column-major run-length encoding. An empty mask is written as a single space.

In [3]:
def rle_encode(mask):
    """Binary (H, W) mask -> 1-based, column-major RLE string. Empty mask -> " "."""
    flat = np.asarray(mask, dtype=np.uint8).flatten(order="F")
    padded = np.concatenate([[0], flat, [0]])
    changes = np.where(padded[1:] != padded[:-1])[0] + 1   # 1-based run boundaries
    changes[1::2] -= changes[::2]                          # (start, end) -> (start, length)
    return " ".join(map(str, changes)) if len(changes) else " "

## 3. Model definition
Same architecture as in training (it must match the checkpoint exactly).

In [4]:
class DoubleConv(nn.Module):
    """(Conv3x3 -> BatchNorm -> ReLU) x 2 : the basic U-Net building block."""
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, 3, padding=1, bias=False), nn.BatchNorm2d(out_ch), nn.ReLU(inplace=True),
            nn.Conv2d(out_ch, out_ch, 3, padding=1, bias=False), nn.BatchNorm2d(out_ch), nn.ReLU(inplace=True))
    def forward(self, x):
        return self.block(x)

class UNetScratch(nn.Module):
    """U-Net with random initialisation (no pretrained weights). Input (B,1,H,W) -> logits (B,4,H,W)."""
    def __init__(self, in_ch=1, n_classes=4, base=32):
        super().__init__()
        c = [base, base * 2, base * 4, base * 8, base * 16]           # 32, 64, 128, 256, 512
        self.enc1, self.enc2 = DoubleConv(in_ch, c[0]), DoubleConv(c[0], c[1])
        self.enc3, self.enc4 = DoubleConv(c[1], c[2]), DoubleConv(c[2], c[3])
        self.pool = nn.MaxPool2d(2)
        self.bottleneck = DoubleConv(c[3], c[4])
        self.up4, self.dec4 = nn.ConvTranspose2d(c[4], c[3], 2, stride=2), DoubleConv(c[3] * 2, c[3])
        self.up3, self.dec3 = nn.ConvTranspose2d(c[3], c[2], 2, stride=2), DoubleConv(c[2] * 2, c[2])
        self.up2, self.dec2 = nn.ConvTranspose2d(c[2], c[1], 2, stride=2), DoubleConv(c[1] * 2, c[1])
        self.up1, self.dec1 = nn.ConvTranspose2d(c[1], c[0], 2, stride=2), DoubleConv(c[0] * 2, c[0])
        self.head = nn.Conv2d(c[0], n_classes, kernel_size=1)         # 4 independent logits per pixel

    def forward(self, x):
        e1 = self.enc1(x)                       # (B, 32, 128, 800)
        e2 = self.enc2(self.pool(e1))           # (B, 64,  64, 400)
        e3 = self.enc3(self.pool(e2))           # (B,128,  32, 200)
        e4 = self.enc4(self.pool(e3))           # (B,256,  16, 100)
        b  = self.bottleneck(self.pool(e4))     # (B,512,   8,  50)
        d4 = self.dec4(torch.cat([self.up4(b),  e4], dim=1))
        d3 = self.dec3(torch.cat([self.up3(d4), e3], dim=1))
        d2 = self.dec2(torch.cat([self.up2(d3), e2], dim=1))
        d1 = self.dec1(torch.cat([self.up1(d2), e1], dim=1))
        return self.head(d1)                    # raw logits, no sigmoid here

## 4. Test dataset
Same normalisation as training; no augmentation.

In [5]:
def augment_pair(img, masks):
    """img (H,W), masks (4,H,W). Geometric ops hit both; brightness/contrast only the image."""
    if random.random() < 0.5:                       # horizontal flip
        img, masks = img[:, ::-1], masks[:, :, ::-1]
    if random.random() < 0.5:                       # vertical flip
        img, masks = img[::-1, :], masks[:, ::-1, :]
    contrast = random.uniform(0.8, 1.2)
    brightness = random.uniform(-0.1, 0.1)
    img = np.clip(img * contrast + brightness, 0.0, 1.0)
    return img, masks

class SurfaceDataset(Dataset):
    """Returns (image[1,H,W] float, mask[4,H,W] float). masks=None for test images."""
    def __init__(self, image_ids, image_index, rle_table=None, augment=False, mean=0.5, std=0.25):
        self.ids = list(image_ids)
        self.index = image_index
        self.rows = rle_table.loc[self.ids, DEFECT_TYPES].values if rle_table is not None else None
        self.augment, self.mean, self.std = augment, mean, std

    def __len__(self):
        return len(self.ids)

    def __getitem__(self, i):
        img = np.asarray(Image.open(self.index[self.ids[i]]).convert("L"), dtype=np.float32) / 255.
        assert img.shape == (IMG_H, IMG_W), f"unexpected image shape {img.shape}"
        if self.rows is not None:
            masks = np.stack([rle_decode(r) for r in self.rows[i]]).astype(np.float32)
        else:
            masks = np.zeros((len(DEFECT_TYPES), IMG_H, IMG_W), dtype=np.float32)
        if self.augment:
            img, masks = augment_pair(img, masks)
        img = (img - self.mean) / self.std
        return (torch.from_numpy(np.ascontiguousarray(img))[None],
                torch.from_numpy(np.ascontiguousarray(masks)))

## 5. Files and checkpoint

In [6]:
def find_one(patterns):
    for p in patterns:
        hits = glob.glob(p, recursive=True)
        if hits:
            return sorted(hits)[0]
    raise FileNotFoundError(f"None of {patterns} matched")

SAMPLE_CSV = find_one(["/kaggle/input/**/sample_submission.csv"])
CKPT_PATH  = find_one(["/kaggle/input/**/best_unet_scratch.pt"])
print(SAMPLE_CSV); print(CKPT_PATH)

IMG_EXT = (".png", ".jpg", ".jpeg", ".bmp", ".tif", ".tiff")
def build_image_index(root="/kaggle/input"):
    index = {}
    for dirpath, _, files in os.walk(root):
        for f in files:
            if f.lower().endswith(IMG_EXT):
                index[os.path.splitext(f)[0]] = os.path.join(dirpath, f)
    return index
IMAGE_INDEX = build_image_index()
print("images found:", len(IMAGE_INDEX))

/kaggle/input/competitions/industrial-surface-defect-detection/public/sample_submission.csv
/kaggle/input/notebooks/mruniksurya/milestone-2/best_unet_scratch.pt
images found: 14394


In [7]:
model = UNetScratch().to(DEVICE)
model.load_state_dict(torch.load(CKPT_PATH, map_location=DEVICE))
model.eval()
print("checkpoint loaded; parameters:", f"{sum(p.numel() for p in model.parameters()):,}")

checkpoint loaded; parameters: 7,762,564


## 6. Predict and write the submission

In [8]:
sample_sub = pd.read_csv(SAMPLE_CSV)
test_ids = sorted(sample_sub["image_id"].unique())
print("test images:", len(test_ids), "| rows expected:", len(sample_sub))
assert set(test_ids).issubset(IMAGE_INDEX), "some test image files not found"

test_ds = SurfaceDataset(test_ids[:32] if DEBUG else test_ids, IMAGE_INDEX, None, augment=False, mean=IMG_MEAN, std=IMG_STD)
test_loader = DataLoader(test_ds, batch_size=32, shuffle=False, num_workers=2)

model.eval()
records = []
with torch.no_grad():
    for b, (x, _) in enumerate(test_loader):
        masks = (torch.sigmoid(model(x.to(DEVICE))) > BEST_TAU).cpu().numpy().astype(np.uint8)
        for k in range(masks.shape[0]):
            iid = test_ds.ids[b * test_loader.batch_size + k]
            for c, d in enumerate(DEFECT_TYPES):
                records.append((iid, d, rle_encode(masks[k, c])))
pred = pd.DataFrame(records, columns=["image_id", "defect_type", "mask_rle"])

# Keep the exact row order / ID column of sample_submission.csv
submission = sample_sub[["ID", "image_id", "defect_type"]].merge(pred, on=["image_id", "defect_type"], how="left")
submission["mask_rle"] = submission["mask_rle"].fillna(" ")        # DEBUG runs: missing images -> empty
assert len(submission) == len(sample_sub) and list(submission.columns) == list(sample_sub.columns)
submission.to_csv("/kaggle/working/submission.csv", index=False)
print("empty-prediction rate:", (submission["mask_rle"].str.strip() == "").mean().round(4))
submission.head(8)

test images: 3000 | rows expected: 12000
empty-prediction rate: 0.4803


,ID,image_id,defect_type,mask_rle
0,1,WAFNKCRR7Q,defect_a,
1,2,WAFNKCRR7Q,defect_b,
2,3,WAFNKCRR7Q,defect_c,
3,4,WAFNKCRR7Q,defect_d,
4,5,YZXKEWSUDD,defect_a,
5,6,YZXKEWSUDD,defect_b,
6,7,YZXKEWSUDD,defect_c,
7,8,YZXKEWSUDD,defect_d,65492 8 65589 10 65613 18 65714 17 65737 24 65...


## 7. Sanity checks

In [9]:
sub = pd.read_csv("/kaggle/working/submission.csv", keep_default_na=False)
print(sub.shape, "| expected rows:", len(sample_sub))
print(sub["defect_type"].value_counts().to_dict())
print("empty-prediction rate per class:")
print(sub.assign(empty=sub["mask_rle"].str.strip() == "").groupby("defect_type")["empty"].mean().round(3))

(12000, 4) | expected rows: 12000
{'defect_a': 3000, 'defect_b': 3000, 'defect_c': 3000, 'defect_d': 3000}
empty-prediction rate per class:
defect_type
defect_a    0.551
defect_b    0.606
defect_c    0.456
defect_d    0.308
Name: empty, dtype: float64
